In [2]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt
import math

## Load Processed FDA file

In [99]:
# produced in FDA_01
df_orig = pd.read_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_full_AP_mapped_active_ingredients_with_parent.csv")
df_orig.shape

(49990, 27)

In [100]:
df_orig_info = df_orig[['sponsor_name','year','application_number', 'brand_name', 'active_ingredients_split', 'drug_umls_term_norm', 'clinical_studies', 'indications_first_sent', 'indications_and_usage']]
df_orig_info.head()

,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
0,WATSON LABS,2002,ANDA076194,LISINOPRIL AND HYDROCHLOROTHIAZIDE,HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,[],INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...
1,WATSON LABS,2002,ANDA076194,LISINOPRIL AND HYDROCHLOROTHIAZIDE,LISINOPRIL,Lisinopril,[],INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...
2,ROCKWELL MEDCL,2003,ANDA076206,CALCITRIOL,CALCITRIOL,Calcitriol,NaN,NaN,NaN
3,APOTEX,2004,ANDA076212,CARBIDOPA AND LEVODOPA,CARBIDOPA,Carbidopa,NaN,NaN,NaN
4,APOTEX,2004,ANDA076212,CARBIDOPA AND LEVODOPA,LEVODOPA,Levodopa,NaN,NaN,NaN


In [101]:
df_orig[df_orig["active_ingredients_split"].str.contains("cariprazine", case=False, na=False)]


,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
31828,NDA204370,ABBVIE,ORIG,AP,20150917,Type 1 - New Molecular Entity,2015,VRAYLAR,CAPSULE,ORAL,...,CARIPRAZINE,"Allergan, Inc.",CARIPRAZINE,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,[],"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR",CARIPRAZINE HYDROCHLORIDE,cariprazine hydrochloride,C4058574
31829,NDA204370,ABBVIE,ORIG,AP,20150917,Type 1 - New Molecular Entity,2015,VRAYLAR,CAPSULE,ORAL,...,CARIPRAZINE,"Allergan, Inc.",CARIPRAZINE,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,[],"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR",CARIPRAZINE,Cariprazine,C2936870
36518,ANDA213932,SUN PHARM,ORIG,AP,20220930,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE HYDROCHLORIDE,cariprazine hydrochloride,C4058574
36520,ANDA213984,ZYDUS,ORIG,AP,20220909,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE HYDROCHLORIDE,cariprazine hydrochloride,C4058574
48091,NDA204370,ABBVIE,ORIG,AP,20150917,Type 1 - New Molecular Entity,2015,VRAYLAR,CAPSULE,ORAL,...,CARIPRAZINE,"Allergan, Inc.",CARIPRAZINE,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,[],"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR",CARIPRAZINE HYDROCHLORIDE,Cariprazine,C2936870
49545,ANDA213932,SUN PHARM,ORIG,AP,20220930,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE HYDROCHLORIDE,Cariprazine,C2936870
49546,ANDA213984,ZYDUS,ORIG,AP,20220909,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE HYDROCHLORIDE,Cariprazine,C2936870


In [102]:
df_orig[df_orig["active_ingredients_substance_brand"].str.contains("morphine", case=False, na=False)]


,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
1267,ANDA203518,TRIS PHARMA INC,ORIG,AP,20150512,NaN,2015,MORPHINE SULFATE,SOLUTION,ORAL,...,MORPHINE SULFATE,Bryant Ranch Prepack,MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,[],MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
1346,NDA204223,FRESENIUS KABI USA,ORIG,AP,20131030,Type 5 - New Formulation or New Manufacturer,2013,MORPHINE SULFATE,SOLUTION,"INTRAMUSCULAR, INTRAVENOUS",...,MORPHINE SULFATE,"Fresenius Kabi, USA LLC",MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine Sulfate Injec...,1 INDICATIONS AND USAGE Morphine Sulfate Injec...,[],MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
1589,ANDA206420,RHODES PHARMS,ORIG,AP,20160712,NaN,2016,MORPHINE SULFATE,SOLUTION,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
1660,ANDA207270,DR REDDYS LABS SA,ORIG,AP,20220112,NaN,2022,MORPHINE SULFATE,TABLET,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
2210,ANDA212451,ALKEM LABS LTD,ORIG,AP,20201203,NaN,2020,MORPHINE SULFATE,TABLET,ORAL,...,MORPHINE SULFATE,"Ascend Laboratories, LLC",MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine sulfate table...,1 INDICATIONS AND USAGE Morphine sulfate table...,[],MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49420,NDA019999,MERIDIAN MEDCL TECHN,ORIG,AP,19900712,Type 5 - New Formulation or New Manufacturer,1990,MORPHINE SULFATE (AUTOINJECTOR),SOLUTION,INTRAMUSCULAR,...,NaN,NaN,NaN,NaN,NaN,NaN,"MORPHINE SULFATE, MORPHINE SULFATE (AUTOINJECTOR)",MORPHINE SULFATE,Morphine,C0026549
49718,NDA021671,PACIRA PHARMS INC,ORIG,AP,20040518,Type 3 - New Dosage Form,2004,DEPODUR,"INJECTABLE, LIPOSOMAL",EPIDURAL,...,NaN,NaN,NaN,NaN,NaN,NaN,"MORPHINE SULFATE, DEPODUR",MORPHINE SULFATE,Morphine,C0026549
49866,ANDA071052,FRESENIUS KABI USA,ORIG,AP,19861007,NaN,1986,ASTRAMORPH PF,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,"MORPHINE SULFATE, ASTRAMORPH PF",MORPHINE SULFATE,Morphine,C0026549
49930,ANDA074769,RHODES PHARMS,ORIG,AP,19980702,NaN,1998,MORPHINE SULFATE,"TABLET, EXTENDED RELEASE",ORAL,...,MORPHINE SULFATE,Rhodes Pharmaceuticals L. P.,MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine sulfate exten...,1 INDICATIONS AND USAGE Morphine sulfate exten...,[],MORPHINE SULFATE,MORPHINE SULFATE,Morphine,C0026549


## Load DS drugs

In [103]:
terms_path = "out/unique_drug_terms_292514.csv" # from Translation_01_Drug
terms = pd.read_csv(terms_path)

In [104]:
terms_for_details = terms[terms["merged_umls_label"].str.len() > 2]
terms_for_details = terms_for_details[terms_for_details['n_articles']>1]

terms_for_details.head()

,merged_umls_label,n_articles
0,Dexamethasone,5806
1,Acetylcysteine,4559
2,NG-Nitroarginine Methyl Ester,4306
3,Sirolimus,4107
4,Doxorubicin,4021


In [105]:
terms_for_details.shape

(74547, 2)

In [106]:
terms_for_details[terms_for_details["merged_umls_label"].str.contains("zoloft", case=False, na=False)].head()

,merged_umls_label,n_articles


## Merge drugs from dataset with FDA details

In [107]:
# 1) Build normalized join keys (upper + strip)
df_counts = terms_for_details.copy()
df_exploded = df_orig_info.copy()

df_counts["__key"] = df_counts["merged_umls_label"].astype(str).str.upper().str.strip()
df_exploded["__key"] = df_exploded["drug_umls_term_norm"].astype(str).str.upper().str.strip()

# 2) Left-merge ON the ingredient (keep all rows from df_exploded)
df_merged = df_counts.merge(
    df_exploded,
    on="__key",
    how="left"
)

# 3) Clean up
df_merged = df_merged.drop(columns="__key")
df_merged.head()

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
0,Dexamethasone,5806,SUN PHARM INDUSTRIES,1974.0,ANDA084013,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
1,Dexamethasone,5806,WHITEWORTH TOWN PLSN,1975.0,ANDA084327,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
2,Dexamethasone,5806,WATSON LABS,1977.0,ANDA085455,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN


In [108]:
df_ds_drugs_with_FDA_info = df_merged[df_merged["application_number"].notna()]
df_ds_drugs_with_FDA_info.head()

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
0,Dexamethasone,5806,SUN PHARM INDUSTRIES,1974.0,ANDA084013,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
1,Dexamethasone,5806,WHITEWORTH TOWN PLSN,1975.0,ANDA084327,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
2,Dexamethasone,5806,WATSON LABS,1977.0,ANDA085455,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN


In [109]:
df_ds_drugs_with_FDA_info.merged_umls_label.nunique()

2745

In [110]:
df_ds_drugs_with_FDA_info.to_csv("out/df_ds_drugs_with_FDA_info.csv", index=False)
df_ds_drugs_with_FDA_info.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/df_ds_drugs_with_FDA_info.csv", index=False)


In [111]:
df_with_indications = df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info["indications_first_sent"].notna()]
df_with_indications['unique_id'] = (
    df_with_indications['merged_umls_label']
    .str.replace(" ", "_")
    .str.lower()
    +"_" + df_with_indications['application_number'].astype(str)
)


/tmp/ipykernel_2820109/2705641560.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_indications['unique_id'] = (


In [112]:
df_with_indications.head()

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage,unique_id
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...,dexamethasone_ANDA088248
5,Dexamethasone,5806,SANDOZ,1988.0,NDA050592,TOBRADEX,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Tobramycin and dexametha...,INDICATIONS AND USAGE Tobramycin and dexametha...,dexamethasone_NDA050592
6,Dexamethasone,5806,PADAGIS US,1989.0,ANDA062938,NEOMYCIN AND POLYMYXIN B SULFATES AND DEXAMETH...,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE: For steroid-responsive ...,INDICATIONS AND USAGE: For steroid-responsive ...,dexamethasone_ANDA062938
7,Dexamethasone,5806,BAUSCH AND LOMB,1995.0,ANDA064135,DEXASPORIN,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE For steroid-responsive i...,INDICATIONS AND USAGE For steroid-responsive i...,dexamethasone_ANDA064135
8,Dexamethasone,5806,PRASCO,1971.0,ANDA080399,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...,dexamethasone_ANDA080399


In [113]:
df_with_indications[df_with_indications["merged_umls_label"].str.contains("cariprazine", case=False, na=False)]

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage,unique_id
28150,Cariprazine,35,ABBVIE,2015.0,NDA204370,VRAYLAR,CARIPRAZINE,Cariprazine,[],1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,cariprazine_NDA204370
28151,Cariprazine,35,ABBVIE,2015.0,NDA204370,VRAYLAR,CARIPRAZINE HYDROCHLORIDE,Cariprazine,[],1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,cariprazine_NDA204370


In [114]:
df_with_indications.shape

(19901, 12)

In [115]:
df_with_indications.merged_umls_label.nunique()

2201

### save for LLM processing

In [116]:
df_with_indications.to_csv("out/drugs_with_indications.csv", index=False)
df_with_indications.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/drugs_with_indications.csv", index=False)


In [49]:
# Calculate chunk size
num_chunks = 3
chunk_size = math.ceil(len(df_with_indications) / num_chunks)

# Split and save each chunk
for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = start_idx + chunk_size
    chunk_df = df_with_indications.iloc[start_idx:end_idx]
    print(f"chunk size {chunk_df.shape}")
    chunk_df.to_csv(f"out/drugs_with_indications_chunk_{i+1}.csv", index=False)


chunk size (4255, 12)
chunk size (4255, 12)
chunk size (4254, 12)


In [50]:
df_ds_drugs_with_FDA_info.merged_umls_label.nunique()

1673

In [51]:
df_filtered = df_ds_drugs_with_FDA_info[
    (df_ds_drugs_with_FDA_info["active_ingredients_split"] == "DEXAMETHASONE") &
    (df_ds_drugs_with_FDA_info["application_number"].astype(str).str.startswith("NDA"))
]
df_filtered.head() 

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
5,Dexamethasone,5806,SANDOZ,1988.0,NDA050592,TOBRADEX,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Tobramycin and dexametha...,INDICATIONS AND USAGE Tobramycin and dexametha...
19,Dexamethasone,5806,ABBVIE,2009.0,NDA022315,OZURDEX,DEXAMETHASONE,Dexamethasone,[],1 INDICATIONS AND USAGE OZURDEX ® is a cortico...,1 INDICATIONS AND USAGE OZURDEX ® is a cortico...
20,Dexamethasone,5806,ASPEN GLOBAL INC,1960.0,NDA012674,HEXADROL,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
21,Dexamethasone,5806,ASPEN GLOBAL INC,1962.0,NDA012675,HEXADROL,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
22,Dexamethasone,5806,OCULAR THERAPEUTIX,2018.0,NDA208742,DEXTENZA,DEXAMETHASONE,Dexamethasone,"['NCT02034019', 'NCT02089113', 'NCT02736175', ...",1 INDICATIONS AND USAGE DEXTENZA ® is a cortic...,1 INDICATIONS AND USAGE DEXTENZA ® is a cortic...
23,Dexamethasone,5806,HARROW EYE,2009.0,NDA050818,TOBRADEX ST,DEXAMETHASONE,Dexamethasone,[],1 INDICATIONS AND USAGE TOBRADEX ST ophthalmic...,1 INDICATIONS AND USAGE TOBRADEX ST ophthalmic...
36,Dexamethasone,5806,HARROW EYE,1963.0,NDA050023,MAXITROL,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE For steroid-responsive i...,INDICATIONS AND USAGE For steroid-responsive i...
37,Dexamethasone,5806,NOVARTIS,1988.0,NDA050616,TOBRADEX,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE: TOBRADEX (tobramycin an...,INDICATIONS AND USAGE: TOBRADEX (tobramycin an...
40,Dexamethasone,5806,DEXCEL,2019.0,NDA211379,HEMADY,DEXAMETHASONE,Dexamethasone,[],1 INDICATIONS AND USAGE HEMADY is indicated in...,1 INDICATIONS AND USAGE HEMADY is indicated in...


In [52]:
df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info['active_ingredients_split']=="RITUXIMAB"].indications_first_sent.iloc[0]

"1 INDICATIONS AND USAGE RITUXAN is a CD20-directed cytolytic antibody indicated for the treatment of: Adult patients with Non-Hodgkin's Lymphoma (NHL) ( 1.1 )"

In [53]:
df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info['active_ingredients_split']=="RITUXIMAB"].indications_first_sent.iloc[1]

'1 INDICATIONS AND USAGE RITUXAN HYCELA is a combination of rituximab, a CD20-directed cytolytic antibody, and hyaluronidase human, an endoglycosidase, indicated for the treatment of adult patients with: Follicular Lymphoma (FL) ( 1.1 ) Relapsed or refractory, follicular lymphoma as a single agent Previously untreated follicular lymphoma in combination with first line chemotherapy and, in patients achieving a complete or partial response to rituximab in combination with chemotherapy, as single-agent maintenance therapy Non-progressing (including stable disease), follicular lymphoma as a single agent after first-line cyclophosphamide, vincristine, and prednisone (CVP) chemotherapy Diffuse Large B-cell Lymphoma (DLBCL) ( 1.2 ) Previously untreated diffuse large B-cell lymphoma in combination with cyclophosphamide, doxorubicin, vincristine, prednisone (CHOP) or other anthracycline-based chemotherapy regimens Chronic Lymphocytic Leukemia (CLL) ( 1.3 ) Previously untreated and previously tr